# Solar Domain Baseline — solar qr


## 1. Setup



In [1]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed()

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements").exists() and (candidate / "solar").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside the cloned repo."
    )

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "solar"
DATA_PATH = ROOT / "data" / "solardata_modified.csv"
PRED_DIR = ROOT / "outputs" / "predictions"
METRIC_DIR = ROOT / "outputs" / "metrics"

PRED_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Solar root:", ROOT)
print("Data:", DATA_PATH)

# Shared solar utilities
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from src.metrics import compute_metrics, print_metrics
from power_conversion import convert_solar_result_to_power


import lightgbm as lgb
from sklearn.cluster import KMeans

In [2]:
TEST_START = pd.Timestamp("2023-10-01")
DAY_HOURS = range(8, 19)

MAIN_QUANTILES = [round(q, 1) for q in np.arange(0.1, 1.0, 0.1)]
TAIL_QUANTILES = [0.01, 0.025, 0.05, 0.95, 0.975, 0.99]
QUANTILES = sorted(set(MAIN_QUANTILES + TAIL_QUANTILES))

OUTPUT_PATH = PRED_DIR / "qgb_quantile.csv"
METRICS_PATH = METRIC_DIR / "qgb_quantile.json"
POWER_OUTPUT_PATH = PRED_DIR / "qgb_quantile_power.csv"
POWER_METRICS_PATH = METRIC_DIR / "qgb_quantile_power.json"


## 2. Load data

In [3]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

print(df.shape)
df.head()

## 3. Feature engineering


In [6]:
WEATHER_LAG24 = [
    "AIRTEMP_lag24", "RH_AVG_lag24", "DEWPT_lag24",
    "WS_lag24", "GHI_lag24", "DNI_lag24", "DIFF_lag24",
]

def build_features(df):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    out = out.sort_values(["item_id", "timestamp"]).copy()

    out["hour"] = out["timestamp"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)

    for lag in (24, 48, 168):
        out[f"POA_lag{lag}"] = out.groupby("item_id")["target"].shift(lag)

    out["POA_diff_24"] = out["POA_lag24"] - out["POA_lag48"]
    out["GHI_diff_24"] = (
        out["GHI_lag24"]
        - out.groupby("item_id")["GHI_lag24"].shift(24)
    )
    out["clearsky_ratio"] = out["POA_lag24"] / (out["GHI_lag24"] + 1e-6)

    lagged_target = out.groupby("item_id")["target"].shift(24)
    out["POA_roll_mean_3d"] = (
        lagged_target.groupby(out["item_id"])
        .rolling(72, min_periods=72)
        .mean()
        .reset_index(level=0, drop=True)
    )
    out["POA_roll_std_3d"] = (
        lagged_target.groupby(out["item_id"])
        .rolling(72, min_periods=72)
        .std()
        .reset_index(level=0, drop=True)
    )
    return out

FEATURE_COLS = [
    "POA_lag24", "POA_lag48", "POA_lag168",
    *WEATHER_LAG24,
    "POA_diff_24", "GHI_diff_24", "clearsky_ratio",
    "POA_roll_mean_3d", "POA_roll_std_3d",
    "hour_sin", "hour_cos",
]

missing = [c for c in WEATHER_LAG24 if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected lag-24 weather columns: {missing}")

features = build_features(df)

## 4. model

In [7]:
class RBFTransformer:
    def __init__(self, n_centers=20, random_state=SEED):
        self.n_centers = n_centers
        self.random_state = random_state

    def fit(self, X):
        n_centers = min(self.n_centers, len(X))
        self.kmeans = KMeans(
            n_clusters=n_centers,
            random_state=self.random_state,
            n_init=10,
        )
        self.kmeans.fit(X)
        self.centers = self.kmeans.cluster_centers_

        dists = np.linalg.norm(
            self.centers[:, None, :] - self.centers[None, :, :],
            axis=2,
        )
        positive = dists[dists > 0]
        fallback = np.median(positive) if len(positive) else 1.0
        sigmas = np.median(dists, axis=1)
        self.sigmas = np.where(sigmas > 0, sigmas, fallback)
        return self

    def transform(self, X):
        cols = []
        for center, sigma in zip(self.centers, self.sigmas):
            dist2 = np.sum((X - center) ** 2, axis=1)
            cols.append(np.exp(-dist2 / (2 * sigma**2 + 1e-8)))
        return np.column_stack(cols)


def fit_qgb(train_df, feature_cols, quantiles):
    X = train_df[feature_cols].to_numpy()
    y = train_df["target"].to_numpy()

    rbf = RBFTransformer(n_centers=20).fit(X)
    X_full = np.hstack([X, rbf.transform(X)])

    models = {}
    for q in quantiles:
        model = lgb.LGBMRegressor(
            objective="quantile",
            alpha=q,
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=SEED,
            verbosity=-1,
        )
        model.fit(X_full, y)
        models[q] = model

    return models, rbf


def predict_qgb(models, rbf, df, feature_cols, quantiles):
    X = df[feature_cols].to_numpy()
    X_full = np.hstack([X, rbf.transform(X)])

    pred = df[["timestamp", "item_id"]].copy()
    for q in quantiles:
        pred[str(q)] = models[q].predict(X_full)

    # prevent quantile crossing
    q_cols = [str(q) for q in quantiles]
    pred[q_cols] = np.sort(pred[q_cols].to_numpy(), axis=1)
    pred[q_cols] = pred[q_cols].clip(lower=0.0)
    return pred

## 5. Train/Test

In [4]:
required_cols = FEATURE_COLS + ["target"]

daylight = features[features["hour"].isin(DAY_HOURS)].copy()
daylight = daylight.dropna(subset=required_cols).reset_index(drop=True)

train_df = daylight[daylight["timestamp"] < TEST_START].copy()
test_day_df = daylight[daylight["timestamp"] >= TEST_START].copy()

print("Train rows:", len(train_df))
print("Test daylight rows:", len(test_day_df))

models, rbf = fit_qgb(train_df, FEATURE_COLS, QUANTILES)
day_pred = predict_qgb(models, rbf, test_day_df, FEATURE_COLS, QUANTILES)

## 6. convert back to 24-hour df

In [5]:
def restore_full_test_grid(full_df, day_pred, test_start, quantiles):
    full = full_df.copy()
    full["timestamp"] = pd.to_datetime(full["timestamp"])

    full = full[full["timestamp"] >= test_start][
        ["timestamp", "item_id", "target"]
    ].copy()

    q_cols = [str(q) for q in quantiles]
    out = full.merge(
        day_pred[["timestamp", "item_id", *q_cols]],
        on=["timestamp", "item_id"],
        how="left",
    )

    for col in q_cols:
        out[col] = out[col].fillna(0.0).clip(lower=0.0)

    out["prediction"] = out["0.5"]
    return out.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

final_df = restore_full_test_grid(df, day_pred, TEST_START, QUANTILES)
final_df.head()

## 7. Evaluation


In [6]:

poa_metrics = print_metrics(
    final_df,
    quantiles=MAIN_QUANTILES,
    label="Solar POA",
)

# power conversion
power_df = convert_solar_result_to_power(
    final_df,
    df,
    temperature_col="AIRTEMP",
)

power_metrics = print_metrics(
    power_df,
    quantiles=MAIN_QUANTILES,
    label="Solar power",
)


## 8. save

In [7]:
final_df.to_csv(OUTPUT_PATH, index=False)

